## Notebook 3 - Silver Layer Notebook

### Load Bronze Tables

In [ ]:
from pyspark.sql.functions import col

transactions = spark.table("retail_job_lab.bronze.transactions")
stores = spark.table("retail_job_lab.bronze.stores")
products = spark.table("retail_job_lab.bronze.products")

### Cleanse the Loaded Bronze Tables

In [ ]:
silver_clean_df = transactions \
    .filter(col("status") == "Completed") \
    .dropna(subset=["price_usd"]) \
    .withColumn("quantity", col("quantity").cast("int")) \
    .withColumn("price_usd", col("price_usd").cast("double")) \
    .withColumn("total_amount", col("quantity") * col("price_usd"))

### Perform `Joins` for Data Enrichment

In [ ]:
silver_enriched_df = silver_clean_df \
    .join(stores, "store_id", "left") \
    .join(products, "product_id", "left")

### Write Silver Tables

In [ ]:
silver_clean_df.write.mode("overwrite") \
    .saveAsTable("retail_job_lab.silver.transactions_clean")

silver_enriched_df.write.mode("overwrite") \
    .saveAsTable("retail_job_lab.silver.transactions_enriched")

print("Silver tables created")

### Validate Silver Table

In [ ]:
display(spark.table("retail_job_lab.silver.transactions_enriched"))